# 1) Importar as bibliotecas e os dados

In [13]:
import pandas as pd #trata dos dados
import numpy as np #operações matemáticas
import altair as alt #visualização de dados
from lifelines import CoxPHFitter #análise de sobrevivência

In [14]:
df = pd.read_csv('C:\\Users\\Pedro\\Documents\\GitHub\\estudo\\prog_impatech\\2_ano\\Aplic_Med\\desmame.csv', sep=r'\s+', header=0, names=['id', 'tempo', 'cens', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11'])

print("Dimensões:", df.shape)
df.head()

Dimensões: (150, 14)


,id,tempo,cens,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11
0,1,6.0,1,0,0,0,1,0,0,0,1,1,1,0
1,5,8.0,1,0,0,0,1,1,1,1,1,1,1,1
2,6,0.1,1,1,0,0,0,1,1,0,1,0,0,1
3,8,5.0,1,0,1,0,1,1,0,0,0,0,0,0
4,9,3.0,1,0,0,0,1,1,0,0,1,0,0,0


## a) fazer a separação de grupos e descrição dos mesmos

#

In [15]:
# Separar grupos
censurados = df[df['cens'] == 0]
nao_censurados = df[df['cens'] == 1]

# Tabela descritiva
desc_stats = df.groupby('cens')['tempo'].describe()
print("Estatísticas Descritivas do Tempo (0=Censurado, 1=Observado):")
print(desc_stats)

# Boxplot
boxplot = alt.Chart(df).mark_boxplot().encode(
    x=alt.X('cens:N', title='Status', axis=alt.Axis(labelExpr="datum.value == 0 ? 'Censurado (0)' : 'Desmame (1)'")),
    y=alt.Y('tempo:Q', title='Tempo de Aleitamento'),
    color=alt.Color('cens:N', title='Status', scale=alt.Scale(domain=[0, 1], range=['blue', 'orange']))

).properties(
    title='Distribuição do Tempo de Aleitamento por Status',
    width=300,
    height=300
)

# Densidade
density = alt.Chart(df).transform_density(
    'tempo',
    groupby=['cens'],
    as_=['tempo', 'density']
).mark_line().encode(
    x=alt.X('tempo:Q', title='Tempo'),
    y=alt.Y('density:Q', title='Densidade'),
    color=alt.Color('cens:N', title='Status', scale=alt.Scale(domain=[0, 1], range=['blue', 'orange']))
).properties(
    title='Densidade dos Tempos de Aleitamento',
    width=300,
    height=300
)

# Combinar os gráficos lado a lado
alt.hconcat(boxplot, density).resolve_scale(color='independent')

Estatísticas Descritivas do Tempo (0=Censurado, 1=Observado):
      count      mean       std  min  25%  50%   75%   max
cens                                                      
0      85.0  6.485882  6.074814  0.1  2.0  4.0  10.0  24.0
1      65.0  4.146154  3.625650  0.1  1.0  3.5   5.0  18.0


alt.HConcatChart(...)

Aqui temos uma evidência clara do problema com os dados censurados: Eles não nos trazem exatamente a informação que queremos. Nesse caso, a informação que podemos obter de um bebe censurado com tempo = 8 é que até o momento tempo = 8 o desmame não ocorreu, e mais nada. Enquanto no outro caso podemos de fato ter avaliações concretas de tempo do desmame. E a censura pode ter haver com o desmame, ou não(exemplo: a família pode ter se mudado ou foram retirados por problemas durante o desmame). Por isso, os dados tem que ser usados com delicadeza para que possamos utilizá-los. Pois como à priori eles seriam jogados fora, métodos como cox que conseguem integrá-los, ganham por não "disperdiçar" os dados.

## b) Análise com termos de interação

In [20]:
# Selecionar apenas as covariáveis originais
cols_V = [col for col in df.columns if col.startswith('V')]
df_lasso = df.copy()

# Criar interações manualmente (loop duplo)
interacoes_criadas = []
for i in range(len(cols_V)):
    for j in range(i + 1, len(cols_V)):
        col_i = cols_V[i]
        col_j = cols_V[j]
        nome_interacao = f"{col_i}*{col_j}"
        df_lasso[nome_interacao] = df_lasso[col_i] * df_lasso[col_j]
        interacoes_criadas.append(nome_interacao)

print(f"Total de variáveis preditoras (originais + interações): {len(cols_V) + len(interacoes_criadas)}")

colunas_originais = df_lasso.shape[1]
df_lasso = df_lasso.loc[:, df_lasso.std() > 0]

print(f"Total de colunas após remover constantes: {df_lasso.shape[1]} e total retirado: {colunas_originais - df_lasso.shape[1]}")

# l1_ratio=1.0 define o Lasso puro (seleção de variáveis).
cph_lasso = CoxPHFitter(penalizer=0.1, l1_ratio=1.0)
cph_lasso.fit(df_lasso, duration_col='tempo', event_col='cens')

# Filtramos coeficientes que não foram zerados (magnitude > 1e-5 para evitar ruído numérico)
coeficientes = cph_lasso.params_
selecionadas = coeficientes[abs(coeficientes) > 1e-5].sort_values(ascending=False)

print("\n--- Variáveis Selecionadas pelo Lasso (Coef != 0) ---")
print(selecionadas)

Total de variáveis preditoras (originais + interações): 66
Total de colunas após remover constantes: 68 e total retirado: 1

--- Variáveis Selecionadas pelo Lasso (Coef != 0) ---
covariate
V1*V5    0.600162
V5*V9    0.389964
V7       0.095227
V6*V7    0.030295
Name: coef, dtype: float64


## c) sugestões para análises estatísticas

O método mais comum para isso é o __bootstrap__. Em situações onde não temos muitos dados, eles nos ajudam a fazer essas avaliações estatísticas fazendo B amostras com números estocasticamente pegos da base de dados original. Essas análises são muito importantes para, inclusive, podermos afirmar que um preditor é insignificante para a previsão ou não, pois somente o seu valor nominal não é estatísticamente suficiente para que possamos afirmar se ele é estatisticamente significativo ou não. Para isso, podemos avaliar medidas que levam em conta o erro padrão (como p-valor) ou o intervalo de confiança.